# Дополнительный эксперимент: реальная граница OOM
Запустить на Kaggle T4. Каждый батч проверяется в отдельном процессе. Основная сетка и её калибровка не изменяются.

In [ ]:
from pathlib import Path
import os,sys,subprocess
root=Path("/kaggle/working/hw1_oom");root.mkdir(exist_ok=True);os.chdir(root)

In [ ]:
Path('models.py').write_text('"""Assignment architecture, no BatchNorm specified in the layer table."""\nimport torch\nfrom torch import nn\n\nclass SmallCNN(nn.Sequential):\n    def __init__(self):\n        layers=[]\n        for i,(ci,co,k,s) in enumerate([(3,32,7,2),(32,64,5,1),(64,128,3,2),(128,256,1,1),(256,256,3,2),(256,512,1,1)]):\n            layers += [nn.Conv2d(ci,co,k,stride=s,padding=k//2,bias=False), nn.ReLU(inplace=True)]\n            if i==0: layers += [nn.MaxPool2d(3,2,1)]\n        layers += [nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(512,256),nn.ReLU(inplace=True),nn.Linear(256,100)]\n        super().__init__(*layers)\n')

In [ ]:
Path('equations.py').write_text('"""FP32; MAC=2 FLOPs. Comparisons excluded. All functions broadcast."""\nimport numpy as np\nPARAMETERS = 1040324\nPARAMETER_BYTES = 4*PARAMETERS\n\ndef flops(image_size,batch):\n    s,b=np.broadcast_arrays(np.asarray(image_size,dtype=float),np.asarray(batch,dtype=float))\n    # Convolutions + GAP (n-1 adds and one division) + linear MACs + biases.\n    return b*(17714*s*s+313700)\n\ndef memory(image_size,batch):\n    s,b=np.broadcast_arrays(np.asarray(image_size,dtype=float),np.asarray(batch,dtype=float))\n    # Retained input + first conv activation + pooled output + int64 indices.\n    # Excludes opaque cuDNN workspace and allocator rounding; see derivations.\n    return PARAMETER_BYTES+68*b*s*s\n\ndef layer_costs(image_size,batch):\n    s,b=np.broadcast_arrays(np.asarray(image_size,dtype=float),np.asarray(batch,dtype=float))\n    z=np.zeros_like(s); F=[]; Q=[]\n    prev=3*s*s\n    for i,(ci,co,k,d) in enumerate([(3,32,7,2),(32,64,5,4),(64,128,3,8),(128,256,1,8),(256,256,3,16),(256,512,1,16)]):\n        out=co*(s/d)**2; w=ci*co*k*k\n        F.append(2*b*out*ci*k*k); Q.append(4*(b*(prev+out)+w))\n        F.append(z); Q.append(8*b*out) # separate in-place ReLU read/write\n        prev=out\n        if i==0:\n            out=32*(s/4)**2\n            F.append(z); Q.append(4*b*(prev+out)+8*b*out)\n            prev=out\n    F.append(b*prev); Q.append(4*b*(prev+512))\n    F.append(b*(2*512*256+256)); Q.append(4*(b*(512+256)+512*256+256))\n    F.append(z); Q.append(8*b*256)\n    F.append(b*(2*256*100+100)); Q.append(4*(b*(256+100)+256*100+100))\n    return np.stack(F,axis=-1),np.stack(Q,axis=-1)\n\ndef bytes_moved(image_size,batch):\n    return layer_costs(image_size,batch)[1].sum(axis=-1)\n\ndef latency(image_size,batch,theta):\n    f,q=layer_costs(image_size,batch)\n    return theta[\'launch_seconds\']*f.shape[-1]+np.maximum(f/theta[\'flops_per_second\'],q/theta[\'bytes_per_second\']).sum(axis=-1)\n\ndef energy(image_size,batch,theta_energy):\n    return (theta_energy[\'joules_per_flop\']*flops(image_size,batch)\n            +theta_energy[\'joules_per_byte\']*bytes_moved(image_size,batch)\n            +theta_energy[\'baseline_watts\']*latency(image_size,batch,theta_energy[\'latency\']))\n')

In [ ]:
Path('oom_probe.py').write_text('"""Supplemental real CUDA OOM search; main 132-point dataset is untouched.\nRun: python oom_probe.py. Each configuration uses a fresh subprocess on GPU 0.\n"""\nimport argparse, csv, json, math, os, subprocess, sys\nfrom pathlib import Path\n\n\ndef worker(s, b, destination):\n    import time\n    import torch\n    from models import SmallCNN\n    from equations import memory\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.set_num_threads(2)\n    torch.manual_seed(2026)\n    assert torch.cuda.is_available(), \'A real CUDA GPU is required\'\n    torch.cuda.set_device(0)\n    p = torch.cuda.get_device_properties(0)\n    row = dict(S=s, B=b, status=\'\', memory=None, peak_before_failure=None,\n               memory_prediction=float(memory(s,b)), gpu=p.name,\n               total_memory=p.total_memory, torch=torch.__version__,\n               cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),\n               phase=\'model\', error=\'\', experiment=\'supplemental_oom\',\n               is_validation=1, latency=None, energy=None)\n    model = x = y = None\n    try:\n        with torch.inference_mode():\n            model = SmallCNN().cuda().eval()\n            torch.cuda.synchronize()\n            free, total = torch.cuda.mem_get_info()\n            row[\'free_before_input\'] = free\n            row[\'allocated_before_input\'] = torch.cuda.memory_allocated()\n            # Fixed model is already allocated; compare input/activation estimate\n            # with current free memory, in addition to nominal total capacity.\n            row[\'available_tensor_budget\'] = free + row[\'allocated_before_input\']\n            row[\'predicted_oom_total\'] = int(row[\'memory_prediction\'] > total)\n            row[\'predicted_oom_available\'] = int(row[\'memory_prediction\'] > row[\'available_tensor_budget\'])\n            row[\'phase\'] = \'input\'\n            x = torch.randn(b, 3, s, s, device=\'cuda\')\n            row[\'phase\'] = \'warmup\'\n            for _ in range(3):\n                y = model(x)\n                del y\n            torch.cuda.synchronize()\n            torch.cuda.reset_peak_memory_stats()\n            row[\'phase\'] = \'forward\'\n            t = time.perf_counter()\n            y = model(x)\n            torch.cuda.synchronize()\n            row[\'single_forward_seconds\'] = time.perf_counter() - t\n            row[\'memory\'] = torch.cuda.max_memory_allocated()\n            row[\'status\'] = \'OK\'\n    except torch.cuda.OutOfMemoryError as e:\n        row[\'status\'] = \'OOM\'\n        row[\'peak_before_failure\'] = torch.cuda.max_memory_allocated()\n        row[\'error\'] = str(e)\n    # Any other error propagates; it must never be labelled as OOM.\n    Path(destination).write_text(json.dumps(row, indent=2))\n    print(json.dumps({k:row[k] for k in [\'S\',\'B\',\'status\',\'phase\',\'memory\',\'memory_prediction\']}), flush=True)\n\n\ndef main():\n    out = Path(\'results/oom\');out.mkdir(parents=True,exist_ok=True)\n    rows = []\n    def probe(b, label=\'search\'):\n        path = out/f\'probe_{len(rows):02d}_B{b}.json\'\n        subprocess.run([sys.executable, str(Path(__file__).resolve()), \'--worker\',\n                        \'--batch\',str(b),\'--output\',str(path)], check=True)\n        row = json.loads(path.read_text());row[\'trial\']=len(rows);row[\'purpose\']=label\n        rows.append(row)\n        fields = sorted(set().union(*(x.keys() for x in rows)))\n        with (out/\'measurements_oom.csv\').open(\'w\',newline=\'\') as f:\n            w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)\n        return row\n    # Establish a bracket by actual allocations, without lowering memory limits.\n    low=0;high=256\n    while True:\n        row=probe(high)\n        if row[\'status\']==\'OOM\':break\n        low=high;high*=2\n        if high>8192:raise RuntimeError(\'No OOM found within safety cap; inspect hardware\')\n    initial=[low,high]\n    while high-low>1:\n        mid=(low+high)//2\n        if probe(mid)[\'status\']==\'OK\':low=mid\n        else:high=mid\n    confirmations=[]\n    for _ in range(2):\n        if low:confirmations.append(probe(low,\'boundary_repeat\'))\n        confirmations.append(probe(high,\'boundary_repeat\'))\n    stable=all(x[\'status\']==(\'OK\' if x[\'B\']==low else \'OOM\') for x in confirmations)\n    from equations import PARAMETER_BYTES\n    capacity=rows[0][\'total_memory\']\n    predicted_first=math.floor((capacity-PARAMETER_BYTES)/(68*512**2))+1\n    # Also test the analytic boundary itself; no extrapolated measurement claims.\n    for b in [predicted_first-1,predicted_first]:\n        if b>0 and not any(x[\'B\']==b for x in rows):probe(b,\'analytic_boundary\')\n    summary=dict(S=512,initial_bracket=initial,largest_success_in_search=low,\n                 smallest_oom_in_search=high,boundary_repeats_stable=stable,\n                 predicted_first_oom_total=predicted_first,total_memory=capacity,\n                 gpu=rows[0][\'gpu\'],trials=len(rows),\n                 real_oom_trials=sum(x[\'status\']==\'OOM\' for x in rows),\n                 false_negatives_total=sum(x[\'status\']==\'OOM\' and not x[\'predicted_oom_total\'] for x in rows),\n                 false_positives_total=sum(x[\'status\']==\'OK\' and x[\'predicted_oom_total\'] for x in rows),\n                 protocol=\'One fresh subprocess per trial, CUDA GPU 0, actual OOM only; eval/inference_mode FP32, 3 warmups then one measured forward. No artificial memory cap. No energy/median latency measured in this supplemental experiment.\',\n                 caveat=\'Binary search assumes local monotonic feasibility. The adjacent boundary is repeated; cuDNN algorithm changes can make global feasibility nonmonotonic. Results apply to this session and software.\')\n    (out/\'summary.json\').write_text(json.dumps(summary,indent=2))\n    print(\'OOM EXPERIMENT COMPLETE\',json.dumps(summary),flush=True)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--worker\',action=\'store_true\')\n    parser.add_argument(\'--batch\',type=int);parser.add_argument(\'--output\')\n    args=parser.parse_args()\n    if args.worker:worker(512,args.batch,args.output)\n    else:main()\n')

In [ ]:
Path('analyze_oom.py').write_text('"""Plot the supplemental OOM experiment without fitting the memory equation."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom equations import memory\n\ndef main():\n    out=Path(\'results/oom\')\n    d=pd.read_csv(out/\'measurements_oom.csv\')\n    s=json.loads((out/\'summary.json\').read_text())\n    total=s[\'total_memory\'];lo=s[\'largest_success_in_search\'];hi=s[\'smallest_oom_in_search\']\n    predicted=s[\'predicted_first_oom_total\']\n    success=d[d.status==\'OK\'];failed=d[d.status==\'OOM\']\n    available_first=np.floor((d.available_tensor_budget-4161296)/(68*512**2)).astype(int)+1\n    analysis=dict(predicted_first_oom_available_min=int(available_first.min()),predicted_first_oom_available_max=int(available_first.max()),false_negatives_available=int(((d.status==\'OOM\') & (d.predicted_oom_available==0)).sum()),unique_batches=int(d.B.nunique()),boundary_batch_relative_gap=(predicted-hi)/hi)\n    (out/\'analysis.json\').write_text(json.dumps(analysis,indent=2))\n    bs=np.arange(max(1,int(d.B.min())-40),int(d.B.max())+41)\n    fig,axes=plt.subplots(1,2,figsize=(13,5))\n    ax=axes[0]\n    ax.plot(bs,memory(512,bs)/2**30,label=\'Analytic tensor memory\',color=\'tab:blue\')\n    ax.scatter(success.B,success.memory/2**30,label=\'Measured successful forward\',color=\'black\',zorder=3)\n    ax.axhline(total/2**30,ls=\'--\',color=\'gray\',label=\'GPU total capacity\')\n    ax.axvspan(lo,hi,color=\'tab:red\',alpha=.25,label=\'Observed adjacent OK/OOM bracket\')\n    ax.set(xlabel=\'Batch size B (S=512 pixels)\',ylabel=\'Memory (GiB)\',title=\'Memory model vs successful measurements\')\n    ax.grid(alpha=.2);ax.legend(fontsize=8)\n    ax=axes[1]\n    ax.step(bs,(memory(512,bs)>total).astype(int),where=\'mid\',label=\'Predicted OOM: M > GPU total\')\n    ax.scatter(d.B,(d.status==\'OOM\').astype(int),marker=\'x\',color=\'tab:red\',label=\'Observed status (including repeats)\',zorder=3)\n    ax.axvline(int(available_first.median()),ls=\'--\',color=\'tab:green\',label=f\'Prediction using free budget: {int(available_first.median())}\')\n    ax.axvline(hi,ls=\':\',color=\'tab:red\',label=f\'Observed first OOM in search: {hi}\')\n    ax.axvline(predicted,ls=\'--\',color=\'tab:blue\',label=f\'Analytic first OOM: {predicted}\')\n    ax.set(xlabel=\'Batch size B (S=512 pixels)\',ylabel=\'Forward outcome\',yticks=[0,1],yticklabels=[\'OK\',\'OOM\'],ylim=(-.15,1.2),title=\'OOM boundary detail: measured vs predicted\')\n    ax.set_xlim(lo-12,predicted+12)\n    ax.grid(alpha=.2);ax.legend(fontsize=8,loc=\'center right\')\n    fig.tight_layout();figpath=Path(\'results/figures/oom_boundary.png\');figpath.parent.mkdir(exist_ok=True)\n    fig.savefig(figpath,dpi=170);plt.close(fig)\n    lines=[\'# Дополнительная проверка OOM\',\'\',\n           \'Это отдельный эксперимент за пределами основной сетки: S=512, B>256. Основные 132 измерения и калибровка не изменены.\',\'\',\n           f\'GPU: {s["gpu"]}, полная память {total/2**30:.2f} GiB. Каждый опыт выполнялся в новом процессе на cuda:0 с исходной моделью, FP32, eval(), inference_mode(), тремя прогревами и одним измеряемым forward. Все три флага benchmark/TF32 выключены. Искусственного ограничения памяти и посторонних тензоров не было.\',\'\',\n           f\'Поиск дал соседние точки: **B={lo} — OK; B={hi} — настоящий torch.cuda.OutOfMemoryError**. Обе точки дополнительно проверены дважды; стабильность подтверждена: {s["boundary_repeats_stable"]}.\',\n           f\'Всего {len(d)} запусков ({d.B.nunique()} разных батчей), из них {len(failed)} с OOM.\',\'\',\n           \'| B | Исход | Пик успешного forward, GiB | M(S,B), GiB | Прогноз OOM по полной памяти |\',\'|---:|---|---:|---:|---|\']\n    for b in sorted(d.B.unique()):\n        part=d[d.B==b];state=\'/\'.join(sorted(part.status.unique()));vals=part.memory.dropna()\n        peak=f\'{vals.min()/2**30:.3f}–{vals.max()/2**30:.3f}\' if len(vals)>1 else (f\'{vals.iloc[0]/2**30:.3f}\' if len(vals) else \'—\')\n        lines.append(f\'| {b} | {state} | {peak} | {float(memory(512,b))/2**30:.3f} | {"OOM" if float(memory(512,b))>total else "OK"} |\')\n    lines += [\'\', \'## Сравнение с формулой\', \'\',\n              \'M(512,B)=4161296+68·512²·B байт. Коэффициенты формулы не подгонялись.\',\n              f\'Первый предсказанный OOM по полной памяти: B=floor((GPU_total−4161296)/(68·512²))+1={predicted}. Фактическая соседняя граница поиска — {hi}. Таким образом, формула допускает слишком большой батч.\',\n              f\'Если вместо полной памяти брать свободную память перед входом плюс уже выделенные тензоры модели, предсказанная первая OOM-точка лежит в диапазоне {available_first.min()}–{available_first.max()}; этот вариант также сохранён в CSV.\',\n              f\'На всех пробах (включая повторы): пропущенных OOM по полной памяти — {s["false_negatives_total"]}, ложных OOM — {s["false_positives_total"]}. Это результаты адаптивного поиска, а не независимая оценка частоты ошибок.\', \'\',\n              f\'Относительное смещение первого OOM по полной памяти: ({predicted}−{hi})/{hi}={100*(predicted-hi)/hi:.2f}%. Учёт свободного бюджета уменьшает смещение, но не устраняет его полностью. Это сравнение порога без подгонки формулы.\', \'\',\n              \'Аналитика учитывает живые тензоры и индексы MaxPool, но не неизвестный workspace выбранных cuDNN kernels, округление/фрагментацию аллокатора и расходы вне max_memory_allocated. Поэтому OOM может возникнуть даже при M ниже полной памяти. Сообщения реальных исключений и этап отказа сохранены для каждой пробы в JSON и CSV. У неуспешного прохода полного пика памяти нет: поле memory пустое, peak_before_failure означает лишь максимум до исключения.\', \'\',\n              \'Двоичный поиск использует предположение о локально монотонной вместимости. Повторы подтверждают соседнюю границу в этой сессии, но не доказывают глобальную монотонность: при других размерах cuDNN может выбирать другие алгоритмы. Измерения времени/энергии основной работы не заменяются дополнительными опытами; энергия и медианная latency здесь не измерялись.\', \'\',\n              \'![Память и реальная граница OOM](../figures/oom_boundary.png)\', \'\',\n              \'Воспроизведение из hw1/: `python oom_probe.py`, затем `python analyze_oom.py`.\']\n    (out/\'README.md\').write_text(\'\\n\'.join(lines))\n    print(json.dumps(s,indent=2))\n\nif __name__==\'__main__\':main()\n')

In [ ]:
subprocess.run([sys.executable,"oom_probe.py"],check=True)
subprocess.run([sys.executable,"analyze_oom.py"],check=True)

In [ ]:
from IPython.display import display,Markdown,Image
display(Markdown(Path("results/oom/README.md").read_text()))
display(Image(filename="results/figures/oom_boundary.png"))
import shutil
shutil.make_archive("/kaggle/working/oom-results","zip",root,"results")